### Следующая ячейка - блок импорта общих библиотек и формирования переменных
#### должна выполняться первой

In [ ]:
import os # Отключаем попытку tqdm найти визуальный виджет IProgress 
#from tqdm.auto import tqdm
os.environ['TQDM_DISABLE'] = '1'

In [ ]:
# ИМПОРТ БИБЛИОТЕК
import os
from collections import Counter
from pathlib import Path
import httpx
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
# Загружаем переменные из модуля  ai_engineer_preproject\path.py
from ai_engineer_preproject.path import (
    PROJECT_ROOT, 
    DOCS_DIR, 
    LOCAL_EMBED_MODEL, 
    CHUNK_EMBEDINGS,
    OLLAMA_URL,
    INDEXES,
    LLM_MODEL, 
)

# Получаем путь к папке с документами (данные для LLM)
print(f"Корнева папка проекта:\n{PROJECT_ROOT}")
print(f"Папка с документами датасета:\n{DOCS_DIR}")
print(f"Путь к папке локально сохраненной модели эмбеддингов:\n{LOCAL_EMBED_MODEL}")
print(f"Путь к файлу с локально сохраненными эмбедингами чанков:\n{CHUNK_EMBEDINGS}")
print(f"URL сервера ollama:\n{OLLAMA_URL}")
print(f"Путь к файлу с локально сохраненными индексами:\n{INDEXES}")
print(f"Модель используемая локально:\n{LLM_MODEL}")

In [ ]:
# Готовая функция чанкинга 
def chunk_text(text: str, max_chars: int = 900, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.
    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются
    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [ ]:
# Проверяем работоспособность сервера ollama и моделей
# Отправляем GET-запрос и получаем ответ
response = httpx.get(f"{OLLAMA_URL}/api/tags")
# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")

In [ ]:
# ЗАГРУЖАЕМ ДОКУМЕНТЫ
documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}")

In [ ]:
# НАРЕЗАЕМ НА ЧАНКИ 
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

In [ ]:
# Посмотрим на несколько примеров чанков
for chunk in chunks[0:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

In [ ]:
# СВОДКА ПО ЧАНКАМ
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

source_counts = Counter(c["source"] for c in chunks)

top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков")

In [ ]:
file_path = DOCS_DIR / "api.md"
sample_text = file_path.read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    avg_length = sum(len(c) for c in chunks_sample) // len(chunks_sample) if chunks_sample else 0
    print(f"max_chars = {max_chars}\n-> чанков: {len(chunks_sample)}, средняя длина: {avg_length}")


In [ ]:
# Загрузка модели с сайта HF
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загружаем модель эмбеддингов: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)



In [ ]:
# ===!!!=== ТОЛЬКО ДЛЯ ЗАГРУЗКИ ЛОКАЛЬНО СОХРАНЕННОЙ МОДЕЛИ ЕМБЕДДИНГОВ ===!!!===
embed_model = SentenceTransformer(LOCAL_EMBED_MODEL)

In [ ]:
# ===!!!=== ТОЛЬКО ДЛЯ ЛОКАЛЬНОГО СОХРАНЕНИЯ МОДЕЛИ ЕМБЕДДИНГОВ ===!!!=== 
# folder for save model = {}
# Файловый размер модели около 450 мегабайт
print(f"Сохраняем модель эмбедер в папку:\n{LOCAL_EMBED_MODEL}")
embed_model.save(LOCAL_EMBED_MODEL)

In [ ]:
# ФОРМИРОВАНИЕ ЕМБЕДДИНГОВ
# # Берём только тексты чанков (метаданные оставим в chunks как есть)
texts = [c["text"] for c in chunks]

# Считаем эмбеддинги: на выходе - матрица (N, 384)
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

print(f"Размер матрицы эмбеддингов: {chunk_embeddings.shape}")

In [ ]:
# ЛОКАЛЬНОЕ СОХРАНЕНИЕ ГОТОВЫХ ЕМБЕДДИНГОВ
np.save(CHUNK_EMBEDINGS, chunk_embeddings)

In [ ]:
# ЗАГРУЗКА ЕМЕБЕДДИНГОВ ИЗ ЛОКАЛЬНО СОХРАНЕННОГО ФАЙЛА
chunk_embeddings = np.load(CHUNK_EMBEDINGS)

In [ ]:
print(f"Первый чанк: {chunks[0]['text'][:80]}...")
print(f"Его эмбеддинг (первые 10 чисел): {chunk_embeddings[0][:10]}")
print(f"Длина вектора (норма): {np.linalg.norm(chunk_embeddings[0]):.4f}")

In [ ]:
dim = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"В индексе {index.ntotal} векторов размерности {dim}") 

In [ ]:
print(INDEXES)

In [ ]:
# ===!!!=== ЛОКАЛЬНОЕ СОХРАНЕНИЕ ИНДЕКСОВ FAISS В ФАЙЛ ===!!!===
# Запускать ячейку только для локального сохранение модели индексов
faiss.write_index(index, INDEXES.as_posix())

In [ ]:
# ===!!!=== ЗАГРУЗКА ИНДЕКСОВ FAISS ИЗ ФАЙЛА ===!!!===
# Запускать ячейку только для локальной загрузки модели индексов
index = faiss.read_index(INDEXES.as_posix())

In [ ]:
# Функция для поиска ближайших чанков по смыслу
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    """Возвращает top_k чанков, наиболее близких к запросу по смыслу."""
    # 1. Считаем эмбеддинг запроса
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)

    # 2. Ищем ближайших соседей
    scores, indices = index.search(query_emb, top_k)

    # 3. Собираем результат с метаданными
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

In [ ]:
question = "Что такое Ollama?"


In [ ]:
results = vector_search(question, top_k=3)

print(f"Запрос: {question}\n")
for i, r in enumerate(results, 1):
    print(f"#{i}  [{r['source']}]  score={r['score']:.3f}")
    print(r["text"][:200], "...\n")

In [ ]:
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    results = vector_search(q, top_k=3) 
    for r in results:
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:150].strip()}...\n") 



In [ ]:
print(LLM_MODEL)

In [ ]:
import httpx

response = httpx.post(
    "http://localhost:11434/api/generate",
    json={
        "model": LLM_MODEL,
        "prompt": "Привет! Назови одно интересное свойство Python в одном предложении.",
        "stream": False,
    },
    timeout=120,
)
data = response.json()
print(data["response"])

In [ ]:

def generate_answer(prompt: str) -> str:
    """Отправляет промпт в Ollama и возвращает сгенерированный ответ."""
    response = httpx.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    return data["response"].strip()

In [ ]:
answer = generate_answer("Как сменить папку, в которой Ollama хранит модели?")
print(answer)

In [ ]:
PROMPT_TEMPLATE = """Ты ассистент по документации Ollama.
Отвечай на вопрос пользователя, опираясь только на приведённые ниже фрагменты документации.
Если ответа во фрагментах нет - честно скажи, что не знаешь.
Ответ давай на русском языке.
В конце ответа на отдельной строке укажи источник в формате: Источник: <имя файла>.

Фрагменты документации:
{context}

Вопрос пользователя: {question}

Ответ:"""


def build_prompt(question: str, retrieved: list[dict]) -> str:
    """Собирает промпт из системной инструкции, контекста и вопроса."""
    context_parts = []
    for r in retrieved:
        context_parts.append(f"[файл: {r['source']}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_parts)
    return PROMPT_TEMPLATE.format(context=context, question=question)


question = "Как сменить папку, в которой Ollama хранит модели?"

retrieved = vector_search(question, top_k=3)
prompt = build_prompt(question, retrieved)

print("Промпт:")
print(prompt[:500], "...\n")

print("Ответ модели:")
answer = generate_answer(prompt)
print(answer)

In [ ]:
print(retrieved)